# Universal Drive Uploader - Worker

**A robust, comprehensive downloader that handles ANY file type**

## Features
- 1500+ video sites via yt-dlp (YouTube, Twitter, TikTok, etc.)
- Direct file downloads (.zip, .pdf, .exe, .iso, ANY format)
- Original filename preservation (extracts from Content-Disposition or URL)
- Smart URL detection (auto-detects video sites vs direct files)
- Retry logic with exponential backoff
- Comprehensive error handling

## Instructions
1. Run **Cell 1** to mount Drive and install dependencies
2. Run **Cell 2** to start the worker (runs forever until stopped)

In [ ]:
#@title Cell 1: Setup - Mount Drive & Install Dependencies
#@markdown Run this cell first to set up the environment

import subprocess
import sys

from google.colab import drive

print("="*60)
print("SETUP")
print("="*60)

# Mount Google Drive
print("\nMounting Google Drive...")
drive.mount('/content/drive')
print("Drive mounted successfully")

# Install dependencies
print("\nInstalling dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yt-dlp", "requests"])
print("yt-dlp installed")
print("requests installed")

print("\n" + "="*60)
print("SETUP COMPLETE - Proceed to Cell 2")
print("="*60)

In [ ]:
#@title Cell 2: Worker Loop - Process Downloads
#@markdown Run this cell to start processing downloads from queue

import json
import mimetypes
import os
import re
import time
import urllib.parse
from datetime import datetime

import requests
import yt_dlp
from yt_dlp.utils import DownloadError

# =============================================================================
# CONFIGURATION
# =============================================================================

DRIVE_BASE = "/content/drive/MyDrive"
UPLOADER_DIR = os.path.join(DRIVE_BASE, ".uploader")
QUEUE_FILE = os.path.join(UPLOADER_DIR, "queue.json")
STATUS_FILE = os.path.join(UPLOADER_DIR, "status.json")

POLL_INTERVAL = 5          # Seconds between queue checks
MAX_RETRIES = 3            # Retry attempts for failed downloads
RETRY_DELAY = 5            # Initial retry delay (exponential backoff)
CHUNK_SIZE = 1024 * 1024   # 1MB chunks for direct downloads
REQUEST_TIMEOUT = 30       # Timeout for HTTP requests

# File extensions that should use direct download (NOT yt-dlp)
DIRECT_DOWNLOAD_EXTENSIONS = {
    # Archives
    '.zip', '.rar', '.7z', '.tar', '.gz', '.bz2', '.xz', '.tgz',
    # Documents
    '.pdf', '.doc', '.docx', '.xls', '.xlsx', '.ppt', '.pptx', '.odt', '.ods',
    '.txt', '.rtf', '.csv', '.json', '.xml', '.html', '.htm', '.md',
    # Images
    '.jpg', '.jpeg', '.png', '.gif', '.bmp', '.svg', '.webp', '.ico', '.tiff',
    # Executables & Installers
    '.exe', '.msi', '.dmg', '.pkg', '.deb', '.rpm', '.appimage', '.apk',
    # Disk Images
    '.iso', '.img', '.bin', '.cue',
    # Audio (direct files)
    '.mp3', '.flac', '.wav', '.aac', '.ogg', '.wma', '.m4a',
    # Video (direct files)
    '.mp4', '.mkv', '.avi', '.mov', '.wmv', '.flv', '.webm', '.m4v',
    # Fonts
    '.ttf', '.otf', '.woff', '.woff2',
    # Code & Data
    '.py', '.js', '.css', '.sql', '.db', '.sqlite',
    # Other
    '.torrent', '.nfo', '.srt', '.vtt', '.sub',
}

# Known video platforms (use yt-dlp)
VIDEO_PLATFORMS = [
    'youtube.com', 'youtu.be', 'vimeo.com', 'dailymotion.com',
    'twitter.com', 'x.com', 'facebook.com', 'fb.watch',
    'instagram.com', 'tiktok.com', 'twitch.tv',
    'reddit.com', 'v.redd.it', 'streamable.com',
    'bilibili.com', 'nicovideo.jp', 'soundcloud.com',
]

# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def ensure_uploader_dir():
    """Ensure .uploader directory exists."""
    os.makedirs(UPLOADER_DIR, exist_ok=True)
    if not os.path.exists(QUEUE_FILE):
        with open(QUEUE_FILE, 'w') as f:
            json.dump({"downloads": []}, f, indent=2)
    if not os.path.exists(STATUS_FILE):
        with open(STATUS_FILE, 'w') as f:
            json.dump({"downloads": {}, "last_updated": ""}, f, indent=2)

def load_json(filepath):
    """Load JSON file with error handling."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Warning: Error loading {filepath}: {e}")
        return {"downloads": []} if 'queue' in filepath else {"downloads": {}}

def save_json(filepath, data):
    """Save JSON file with error handling."""
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    except Exception as e:
        print(f"Warning: Error saving {filepath}: {e}")

def update_status(download_id, status_data):
    """Update status for a specific download."""
    status = load_json(STATUS_FILE)
    if "downloads" not in status:
        status["downloads"] = {}
    status["downloads"][download_id] = status_data
    status["last_updated"] = datetime.now().isoformat()
    save_json(STATUS_FILE, status)

def sanitize_filename(filename):
    """Sanitize filename for filesystem compatibility."""
    invalid_chars = '<>:"/\\|?*'
    for char in invalid_chars:
        filename = filename.replace(char, '_')
    filename = ''.join(c for c in filename if ord(c) >= 32)
    name, ext = os.path.splitext(filename)
    if len(name) > 200:
        name = name[:200]
    return name + ext

def get_url_extension(url):
    """Extract file extension from URL."""
    parsed = urllib.parse.urlparse(url)
    path = parsed.path.split('?')[0]
    ext = os.path.splitext(path)[1].lower()
    return ext if ext else None

def is_video_platform(url):
    """Check if URL belongs to a known video platform."""
    parsed = urllib.parse.urlparse(url)
    domain = parsed.netloc.lower().replace('www.', '')
    return any(platform in domain for platform in VIDEO_PLATFORMS)

def should_use_direct_download(url):
    """Determine if URL should use direct download instead of yt-dlp."""
    ext = get_url_extension(url)
    if ext and ext in DIRECT_DOWNLOAD_EXTENSIONS:
        return True
    if not is_video_platform(url):
        try:
            response = requests.head(url, timeout=10, allow_redirects=True)
            content_type = response.headers.get('Content-Type', '').lower()
            if content_type and not any(t in content_type for t in ['text/html', 'application/json']):
                return True
        except Exception:
            pass
    return False

def extract_filename_from_response(response, url):
    """Extract original filename from response headers or URL."""
    filename = None
    content_disp = response.headers.get('Content-Disposition', '')
    if content_disp:
        match = re.search(r"filename\*=(?:UTF-8''|utf-8'')(.+?)(?:;|$)", content_disp, re.IGNORECASE)
        if match:
            filename = urllib.parse.unquote(match.group(1).strip())
        else:
            match = re.search(r'filename=["\']?([^"\';\\n]+)["\']?', content_disp, re.IGNORECASE)
            if match:
                filename = match.group(1).strip()
    if not filename:
        final_url = response.url if response.url else url
        parsed = urllib.parse.urlparse(final_url)
        path = urllib.parse.unquote(parsed.path)
        filename = os.path.basename(path)
        filename = filename.split('?')[0]
    if not filename or filename in ['', 'download', 'file']:
        content_type = response.headers.get('Content-Type', '').split(';')[0]
        ext = mimetypes.guess_extension(content_type) or ''
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"download_{timestamp}{ext}"
    return sanitize_filename(filename)

def format_size(size_bytes):
    """Format bytes to human readable size."""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_bytes < 1024:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024
    return f"{size_bytes:.2f} PB"

def format_speed(speed_bps):
    """Format speed in bytes per second to human readable."""
    return f"{format_size(speed_bps)}/s"

# =============================================================================
# DOWNLOAD FUNCTIONS
# =============================================================================

def download_direct(url, output_path, download_id):
    """Direct file download using requests."""
    print("  [Direct Download] Starting...")
    try:
        response = requests.get(
            url, stream=True, timeout=REQUEST_TIMEOUT, allow_redirects=True,
            headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        )
        response.raise_for_status()
        filename = extract_filename_from_response(response, url)
        filepath = os.path.join(output_path, filename)
        base, ext = os.path.splitext(filepath)
        counter = 1
        while os.path.exists(filepath):
            filepath = f"{base}_{counter}{ext}"
            filename = os.path.basename(filepath)
            counter += 1
        total_size = int(response.headers.get('content-length', 0))
        print(f"  Filename: {filename}")
        if total_size > 0:
            print(f"  Size: {format_size(total_size)}")
        downloaded = 0
        start_time = time.time()
        last_update = start_time
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    current_time = time.time()
                    if current_time - last_update >= 0.5:
                        elapsed = current_time - start_time
                        speed = downloaded / elapsed if elapsed > 0 else 0
                        if total_size > 0:
                            percent = int((downloaded / total_size) * 100)
                            eta = (total_size - downloaded) / speed if speed > 0 else 0
                            eta_str = f"{int(eta)}s" if eta < 60 else f"{int(eta/60)}m {int(eta%60)}s"
                        else:
                            percent = 0
                            eta_str = "Unknown"
                        update_status(download_id, {
                            "status": "downloading", "percent": percent, "speed": format_speed(speed),
                            "downloaded": format_size(downloaded),
                            "total": format_size(total_size) if total_size > 0 else "Unknown",
                            "eta": eta_str, "filename": filename
                        })
                        last_update = current_time
        elapsed = time.time() - start_time
        avg_speed = downloaded / elapsed if elapsed > 0 else 0
        update_status(download_id, {
            "status": "completed", "percent": 100, "speed": format_speed(avg_speed),
            "downloaded": format_size(downloaded), "total": format_size(downloaded), "filename": filename
        })
        print(f"  SUCCESS: {filename} ({format_size(downloaded)})")
        return True, filename
    except requests.exceptions.HTTPError as e:
        error_msg = f"HTTP Error: {e.response.status_code}"
        print(f"  FAILED: {error_msg}")
        return False, error_msg
    except requests.exceptions.ConnectionError as e:
        error_msg = f"Connection Error: {str(e)[:100]}"
        print(f"  FAILED: {error_msg}")
        return False, error_msg
    except requests.exceptions.Timeout:
        print("  FAILED: Request timed out")
        return False, "Request timed out"
    except Exception as e:
        error_msg = f"Error: {str(e)[:100]}"
        print(f"  FAILED: {error_msg}")
        return False, error_msg

def download_with_ytdlp(url, output_path, download_id):
    """Download video/audio using yt-dlp."""
    print("  [yt-dlp] Starting...")
    progress_data = {'filename': 'Unknown', 'last_update': time.time()}

    def progress_hook(d):
        current_time = time.time()
        if current_time - progress_data['last_update'] < 0.5:
            return
        progress_data['last_update'] = current_time
        if d['status'] == 'downloading':
            filename = os.path.basename(d.get('filename', 'Unknown'))
            progress_data['filename'] = filename
            percent_str = d.get('_percent_str', '0%').strip()
            try:
                percent = float(percent_str.replace('%', ''))
            except ValueError:
                percent = 0
            update_status(download_id, {
                "status": "downloading", "percent": int(percent),
                "speed": d.get('_speed_str', 'N/A').strip(),
                "downloaded": d.get('_downloaded_bytes_str', 'N/A').strip(),
                "total": d.get('_total_bytes_str', d.get('_total_bytes_estimate_str', 'N/A')),
                "eta": d.get('_eta_str', 'N/A').strip(), "filename": filename
            })
        elif d['status'] == 'finished':
            filename = os.path.basename(d.get('filename', 'Unknown'))
            progress_data['filename'] = filename
            update_status(download_id, {"status": "processing", "percent": 100, "speed": "N/A", "filename": filename})

    try:
        ydl_opts = {
            'outtmpl': os.path.join(output_path, '%(title)s.%(ext)s'),
            'progress_hooks': [progress_hook], 'format': 'best', 'merge_output_format': 'mp4',
            'quiet': True, 'no_warnings': True, 'extract_flat': False, 'ignoreerrors': False,
            'retries': 3, 'fragment_retries': 3, 'sleep_interval': 1, 'max_sleep_interval': 5,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            if info:
                filename = os.path.basename(ydl.prepare_filename(info))
                title = info.get('title', filename)
                update_status(download_id, {"status": "completed", "percent": 100, "speed": "N/A", "filename": filename})
                print(f"  SUCCESS: {title}")
                return True, filename
            return False, "No video info extracted"
    except DownloadError as e:
        error_msg = f"Download failed: {str(e)[:100]}"
        print(f"  FAILED: {error_msg}")
        return False, error_msg
    except Exception as e:
        error_msg = f"yt-dlp error: {str(e)[:100]}"
        print(f"  FAILED: {error_msg}")
        return False, error_msg

def smart_download(url, output_path, download_id):
    """Smart download that automatically chooses the best method."""
    if should_use_direct_download(url):
        print("  Detected: Direct file download")
        return download_direct(url, output_path, download_id)
    if is_video_platform(url):
        print("  Detected: Video platform")
        return download_with_ytdlp(url, output_path, download_id)
    print("  Detected: Unknown URL type, trying yt-dlp first...")
    success, result = download_with_ytdlp(url, output_path, download_id)
    if success:
        return success, result
    print("  yt-dlp failed, trying direct download...")
    return download_direct(url, output_path, download_id)

# =============================================================================
# MAIN WORKER LOOP
# =============================================================================

def process_download(item, queue):
    """Process a single download item with retry logic."""
    download_id = item['id']
    url = item['url']
    folder = item.get('folder', 'Downloads')
    timestamp = datetime.now().strftime('%H:%M:%S')
    print(f"\n{'='*60}")
    print(f"[{timestamp}] Processing Download")
    print(f"{'='*60}")
    print(f"  ID: {download_id[:8]}...")
    print(f"  URL: {url[:80]}{'...' if len(url) > 80 else ''}")
    print(f"  Folder: {folder}")
    output_path = os.path.join(DRIVE_BASE, folder)
    os.makedirs(output_path, exist_ok=True)
    update_status(download_id, {"status": "starting", "percent": 0, "speed": "N/A", "filename": "Initializing..."})
    for attempt in range(1, MAX_RETRIES + 1):
        if attempt > 1:
            delay = RETRY_DELAY * (2 ** (attempt - 2))
            print(f"  Retry {attempt}/{MAX_RETRIES} in {delay}s...")
            time.sleep(delay)
        success, result = smart_download(url, output_path, download_id)
        if success:
            item['status'] = 'completed'
            item['completed_at'] = datetime.now().isoformat()
            item['filename'] = result
            save_json(QUEUE_FILE, queue)
            print("  DONE")
            return True
    item['status'] = 'failed'
    item['error'] = result
    item['failed_at'] = datetime.now().isoformat()
    save_json(QUEUE_FILE, queue)
    update_status(download_id, {"status": "failed", "percent": 0, "speed": "N/A", "error": result, "filename": f"Error: {result[:50]}"})
    print(f"  FAILED after {MAX_RETRIES} attempts")
    return False

def worker_loop():
    """Main worker loop - continuously processes downloads from queue."""
    ensure_uploader_dir()
    print("\n" + "="*60)
    print("UNIVERSAL DRIVE UPLOADER - WORKER STARTED")
    print("="*60)
    print(f"Queue: {QUEUE_FILE}")
    print(f"Status: {STATUS_FILE}")
    print(f"Poll interval: {POLL_INTERVAL}s")
    print(f"Max retries: {MAX_RETRIES}")
    print("\nReady! Waiting for downloads...")
    print("(Press Stop button or Ctrl+C to halt)\n")
    print("="*60)
    check_count = 0
    downloads_completed = 0
    downloads_failed = 0
    try:
        while True:
            check_count += 1
            queue = load_json(QUEUE_FILE)
            downloads = queue.get('downloads', [])
            pending = [d for d in downloads if d.get('status') == 'pending']
            if pending:
                print(f"\nFound {len(pending)} pending download(s)")
                for item in pending:
                    try:
                        item['status'] = 'in_progress'
                        save_json(QUEUE_FILE, queue)
                        if process_download(item, queue):
                            downloads_completed += 1
                        else:
                            downloads_failed += 1
                    except Exception as e:
                        print(f"  Unexpected error: {e}")
                        item['status'] = 'failed'
                        item['error'] = str(e)
                        save_json(QUEUE_FILE, queue)
                        downloads_failed += 1
                print(f"\nStats: {downloads_completed} completed, {downloads_failed} failed")
            else:
                if check_count % 12 == 0:
                    timestamp = datetime.now().strftime('%H:%M:%S')
                    print(f"[{timestamp}] Worker alive - {downloads_completed} completed, {downloads_failed} failed")
            time.sleep(POLL_INTERVAL)
    except KeyboardInterrupt:
        print("\n" + "="*60)
        print("WORKER STOPPED (User interrupt)")
        print(f"Final stats: {downloads_completed} completed, {downloads_failed} failed")
        print("="*60)

# Start the worker
worker_loop()